In [14]:
from strava2gpx import strava2gpx
from stravalib.client import Client
from joblib import Parallel, delayed

import pandas as pd

import re
import os
import shutil
import time
import tqdm

import geopandas as gpd
from shapely.geometry import Point, Polygon

import gpxpy
import gpxpy.gpx

cred_auth = eval(open("strava_credentials.txt").read())

In [15]:
def read_gpx_file(file_name,path="./export_4778598/activities/"):
    '''
    Reading in individual GPX file using gpxpy package and turn into a dataframe
    for further geographic intersection calculations.
    '''

    gpx_file_path = path + file_name

    with open(gpx_file_path, 'r') as gpx_file:
        # Parse the GPX data
        gpx = gpxpy.parse(gpx_file)

    points_data = []
    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                points_data.append({
                    'time': point.time,
                    'latitude': point.latitude,
                    'longitude': point.longitude,
                    'elevation': point.elevation,
                })

    df = pd.DataFrame.from_records(points_data)
    
    return(df)

def match_to_blocks(gpx_df,block_centroids):
    '''
    Take GPX data from Strava and match to nearest block centorid (using data sourced from DCOpenData)
    Using 450 foot radius as a max for nearest possible match.
    '''

    #geometry = gpd.points_from_xy(block_centroids['LONGITUDE'], block_centroids['LATITUDE'])

    #gdf1 = gpd.GeoDataFrame(block_centroids,geometry = geometry, crs="EPSG:4326")
    gdf1 = gpd.GeoDataFrame(block_centroids, crs="EPSG:4326")

    geometry = gpd.points_from_xy(gpx_df['longitude'], gpx_df['latitude'])
    gdf2 = gpd.GeoDataFrame(gpx_df, geometry = geometry, crs="EPSG:4326")

    gdf1_proj = gdf1.to_crs("EPSG:3857")
    gdf2_proj = gdf2.to_crs("EPSG:3857")

    ## find intersections

    #joined_gdf = gdf2_proj.sjoin_nearest(gdf1_proj, how="inner", max_distance=450,distance_col='distance')
    joined_gdf = gdf2_proj.sjoin_nearest(gdf1_proj, how="inner", max_distance=25,distance_col='distance')

    return(joined_gdf)

def process_gpx_files(gpx_list):
    '''
    Main function to both read and match gpx data to block centroids for paralellization
    '''

    all_data_df = pd.DataFrame()

    for gpx_file in tqdm.tqdm(gpx_list):

        try:

            gpx_df = read_gpx_file(gpx_file)
                
            #block_centroids = pd.read_csv('Block_Centroids.csv')
            block_centroids = gpd.read_file("Street_Centerlines_1999.geojson")

            gpx_matched_df = match_to_blocks(gpx_df,block_centroids=block_centroids)
            gpx_matched_df['file_name'] = gpx_file

            ## sometimes thematching goes haywire at intersections, so we fix this by removing streets that just appear
            ## randomly in the middle of another street

            for i in range(0,2):

                gpx_matched_df['prev_street'] = gpx_matched_df['ST_NAME'].shift(1)
                gpx_matched_df['next_street'] = gpx_matched_df['ST_NAME'].shift(-1)

                gpx_matched_df = gpx_matched_df[(gpx_matched_df['prev_street'] == gpx_matched_df['ST_NAME']) & (gpx_matched_df['next_street'] == gpx_matched_df['ST_NAME'])]

            all_data_df = pd.concat([all_data_df,gpx_matched_df])
            
        except Exception as e:
            print("Error with {}: {}".format(gpx_file,e))

    return(all_data_df)

## Connecting to Strava to pull activities

In [16]:
client = Client()

auth_url = client.authorization_url(
    client_id=cred_auth['CLIENT_ID'],
    redirect_uri="http://localhost:8000/authorization",
    scope=["read", "activity:read_all"]
)

print(auth_url)

https://www.strava.com/oauth/authorize?client_id=32806&redirect_uri=http%3A%2F%2Flocalhost%3A8000%2Fauthorization&approval_prompt=auto&scope=read%2Cactivity%3Aread_all&response_type=code


In [17]:
token_response = client.exchange_code_for_token(client_id=cred_auth['CLIENT_ID'], client_secret=cred_auth['CLIENT_SECRET'], code='bc1efc7d44c2abffb87bae9f699a8ef5a9cb575d')

In [18]:
client_id = cred_auth['CLIENT_ID']
refresh_token = token_response['refresh_token']
client_secret = cred_auth['CLIENT_SECRET']

# create an instance of strava2gpx
s2g = strava2gpx(client_id, client_secret, refresh_token)

# connect to the Strava API
await s2g.connect()

# get a list of all user's Strava activities
activities_list = await s2g.get_activities_list()

Error getting activities: Failed to get activities


Exception: Failed to get activities

In [14]:
export_df = pd.read_csv("export_4778598/activities.csv")

In [80]:
bad_files = []
preserve_export_files = False

start_time = time.time()
iteration_count = 0
time_limit_seconds = 15 * 60

for activity in activities_list:

    file_name = str(activity[1])
    file_exists = False

    if (activity[3] == 'Ride'):

        for old_file in os.listdir("export_4778598/activities/"):
            if file_name in old_file:
                file_exists = True
            if preserve_export_files:
                if int(file_name) in export_df['Activity ID'].to_list():
                    file_exists = True

        if not file_exists:

            ## so time checks to avoid API rate-limit errors

            current_time = time.time()
            elapsed_time = current_time - start_time

            iteration_count += 1

            if (elapsed_time < time_limit_seconds) & (iteration_count >= 200):
                print("API limit hit ... gonna go to sleep for 15 mins")
                time.sleep(15*60)

            try:
                print("Pulling GPX data for: {}".format(str(activity[0])))
                await s2g.write_to_gpx(activity[1],file_name)

                ## moving file to processing directory

                shutil.move(os.path.join('./{}.gpx'.format(file_name)), os.path.join('export_4778598/activities/{}.gpx'.format(file_name)))

            except Exception as e:
                bad_files.append(activity[1]) 

Pulling GPX data for: NCVC Road Camp
GPX file saved successfully.
Pulling GPX data for: Cherry Blossom Madness
GPX file saved successfully.
Pulling GPX data for: Productivity with Mahkah and Mika
GPX file saved successfully.
Pulling GPX data for: Espresso
GPX file saved successfully.
Pulling GPX data for: Checkmated by Bishop
GPX file saved successfully.
Pulling GPX data for: Morning Ride
GPX file saved successfully.
Pulling GPX data for: Short Z2
GPX file saved successfully.
Pulling GPX data for: 3x10
GPX file saved successfully.
Pulling GPX data for: Someone else's SST
GPX file saved successfully.
Pulling GPX data for: Post-Midsouth Existential Crisis
GPX file saved successfully.
Pulling GPX data for: Big Dumb Ride Gravel World Championships
GPX file saved successfully.
Pulling GPX data for: Expo Day + Brutal Winds
GPX file saved successfully.
Pulling GPX data for: Shakin' Out
GPX file saved successfully.
Pulling GPX data for: Tennis Commute
GPX file saved successfully.
Pulling GPX d

In [ ]:
gpx_file_list = []

for file in os.listdir("export_4778598/activities/"):
    if 'gpx' in file:
        gpx_file_list.append(file)

all_data_df = pd.DataFrame()

n_iterations = (os.cpu_count() // 2) - 2 ## this can be played with a bit here, anything higher than 5 tends to clog up compute and your computer may catch on fire
chunk_size = round(len(gpx_file_list) / n_iterations) ## creating the dataframe chunk sizes based on n_iterations -- more iterations --> smaller chunks

try:
    chunks = [gpx_file_list[i:i + chunk_size] for i in range(0, len(gpx_file_list), chunk_size)]
    df_list = chunks[0:n_iterations]
    
except Exception as e:
    print(e)
    df_list = [gpx_file_list]

results = Parallel(n_jobs=n_iterations, prefer="threads")(delayed(process_gpx_files)(gpx_list) for gpx_list in df_list)

all_results = pd.DataFrame()

for i in results:
    all_results = pd.concat([all_results,i])

stats = all_results.groupby(['OBJECTID'])['file_name'].nunique().reset_index()

centroids = pd.read_csv('Street_Centerlines_1999.csv')

centroids_w_stats = pd.merge(centroids,stats,on='OBJECTID',how='left')

centroids_w_stats.to_csv("geocoded_results_20251103.csv",index=False)

  0%|          | 0/128 [00:00<?, ?it/s]



































  1%|          | 1/128 [00:21<40:55, 19.33s/it]




















  2%|▏         | 2/128 [00:40<40:49, 19.44s/it]






































  2%|▏         | 3/128 [01:11<53:04, 25.47s/it]














































  3%|▎         | 4/128 [01:51<1:04:07, 31.02s/it]









































































  4%|▍         | 5/128 [02:43<1:19:08, 38.60s/it]





































  5%|▍         | 6/128 [03:07<1:08:26, 33.66s/it]





















  5%|▌         | 7/128 [03:29<1:00:25, 29.96s/it]



















  6%|▋         | 8/128 [03:39<47:06, 23.56s/it]  



















  7%|▋         | 9/128 [03:57<42:47, 21.57s/it]







































































  8%|▊         | 10/128 [04:50<1:01:55, 31.49s/it]














  9%|▊         | 11/128 [05:01<49:02, 25.15s/it]  
















  9%|▉ 

Error with 14626034128.gpx: Error parsing XML: Premature end of data in tag trkseg line 9, line 9, column 11 (<string>, line 9)


































 27%|██▋       | 34/128 [15:18<38:12, 24.39s/it]



















 27%|██▋       | 35/128 [15:36<34:44, 22.41s/it]




















 28%|██▊       | 36/128 [15:51<30:57, 20.19s/it]




















 29%|██▉       | 37/128 [16:03<27:04, 17.85s/it]


 37%|███▋      | 47/128 [16:08<20:14, 14.99s/it]

Error with 14636810932.gpx: Error parsing XML: Premature end of data in tag trkseg line 9, line 9, column 11 (<string>, line 9)

























 30%|██▉       | 38/128 [16:32<31:49, 21.22s/it]
























 30%|███       | 39/128 [16:50<29:30, 19.90s/it]





























 31%|███▏      | 40/128 [17:13<30:16, 20.65s/it]
















































 32%|███▏      | 41/128 [17:49<35:43, 24.63s/it]







 33%|███▎      | 42/128 [17:56<29:20, 20.47s/it]































 34%|███▎      | 43/128 [18:26<32:54, 23.23s/it]



















 34%|███▍      | 44/128 [18:44<30:02, 21.46s/it]




















 35%|███▌      | 45/128 [19:03<29:06, 21.04s/it]














































 36%|███▌      | 46/128 [19:37<34:05, 24.95s/it]














 37%|███▋      | 47/128 [19:54<29:40, 21.98s/it]

























 38%|███▊      | 48/128 [20:13<28:09, 21.11s/it]













 38%|███▊      | 49/128 [20:27<25:05, 19.06s/it]





















 39%|███▉      | 50/128 [20:44<24:17, 18.69s/it]
















 41%|████      | 52/128 [21

...
















 55%|█████▍    | 70/128 [28:59<17:56, 18.55s/it]





































































 55%|█████▌    | 71/128 [29:53<28:05, 29.57s/it]






































Error with 11212278078.gpx: Error parsing XML: xmlParseEntityRef: no name, line 7, column 13 (<string>, line 7)
























 56%|█████▋    | 72/128 [30:33<30:33, 32.75s/it]
















































 57%|█████▋    | 73/128 [31:04<29:39, 32.35s/it]













 58%|█████▊    | 74/128 [31:15<23:09, 25.73s/it]









































 59%|█████▊    | 75/128 [31:45<23:17, 26.36s/it]






























 59%|█████▉    | 76/128 [32:09<22:53, 26.40s/it]













 60%|██████    | 77/128 [32:17<17:52, 21.04s/it]

























100%|██████████| 128/128 [32:41<00:00, 15.32s/it]












 61%|██████    | 78/128 [32:55<21:39, 25.99s/it]









100%|██████████| 128/128 [32:58<00:00, 15.46s/it]

 62%|██████▏   | 79/128 [33:06<17:40, 21.64s/it]







































 62%|██████▎   | 80/128 [33:36<19:19, 24.16s/it]





 63%|██████▎   | 81/128 [33:47<15:37, 19.94s/it]










 82%|████████▏ | 105/128 [33:52<05:48, 15.15s/it]

Error with 15730227120.gpx: Error parsing XML: EntityRef: expecting ';', line 7, column 20 (<string>, line 7)









 64%|██████▍   | 82/128 [34:04<14:38, 19.09s/it]









 65%|██████▍   | 83/128 [34:15<12:27, 16.61s/it]



 68%|██████▊   | 87/128 [34:36<05:46,  8.46s/it]



 69%|██████▉   | 88/128 [34:41<04:59,  7.49s/it]










 70%|██████▉   | 89/128 [35:01<07:17, 11.21s/it]



 70%|███████   | 90/128 [35:09<06:28, 10.22s/it]



 71%|███████   | 91/128 [35:17<05:53,  9.55s/it]



















 72%|███████▏  | 92/128 [35:47<09:24, 15.69s/it]





 73%|███████▎  | 93/128 [36:02<09:01, 15.47s/it]


 75%|███████▌  | 96/128 [36:19<04:56,  9.28s/it]




 77%|███████▋  | 99/128 [36:41<03:43,  7.71s/it]


 79%|███████▉  | 101/128 [36:50<02:43,  6.05s/it]


 80%|███████▉  | 102/128 [36:55<02:25,  5.60s/it]


 80%|████████  | 103/128 [37:00<02:15,  5.41s/it]





 82%|████████▏ | 105/128 [37:15<02:24,  6.30s/it]


 83%|████████▎ | 106/128 [37:22<02:21,  6.44s/it]


 85%|████████▌ | 109/128 [37:32<01:26,  4.53s/it]


 87%|████████▋ | 111/128 [37:41<01:21,  4.77s/it]








 90%|████████▉ | 1

In [ ]:
# for file in os.listdir():
#     match_text = re.search('[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9].gpx', file)
#     if match_text:
#         new_file = match_text.group()
#         print(new_file)
#         source_path = os.path.join('./', file)
#         destination_path = os.path.join('./export_4778598/activities/', new_file)

#         if not os.path.exists(destination_path):
#             try:
#                 shutil.move(source_path, destination_path)
#             except Exception as e:
#                 print(e)

14738625687.gpx
15006472151.gpx
14654762591.gpx
14645344262.gpx
14719414481.gpx
14874476004.gpx
14946589765.gpx
15883061478.gpx
15721412219.gpx
16067946493.gpx
16099967567.gpx
16079663613.gpx
16226313035.gpx
15944755044.gpx
14884750148.gpx
16255277824.gpx
16236719029.gpx
15792306952.gpx
15936338883.gpx
15744918486.gpx
14677841228.gpx
14688616315.gpx
15677299835.gpx
15902915938.gpx
15823634127.gpx
15835848560.gpx
15858612468.gpx
15868795343.gpx
15914172517.gpx
15946316742.gpx
16014133976.gpx
16095738128.gpx
16164109590.gpx
16173204311.gpx
16205117901.gpx
16235952083.gpx
16245106134.gpx
16275532373.gpx
16286166100.gpx
16306422634.gpx
15756945043.gpx
15780004255.gpx
14797422742.gpx
16090826906.gpx
16154369937.gpx
14641080761.gpx
15813403344.gpx
15853373836.gpx
15699253471.gpx
14844528758.gpx
14865497333.gpx
14917370912.gpx
14936491007.gpx
14967771324.gpx
14989368391.gpx
15009699449.gpx
16143517426.gpx
14794344845.gpx
14700394498.gpx
14772244431.gpx
15877734214.gpx
15926626097.gpx
15891536